# AI as Judge

[G-Eval](https://deepeval.com/docs/metrics-llm-evals) is a framework that uses LLM as a judge to evaluate LLM outputs. The evaluation can be based on any criteria. G-Eval is implemented by a library called [DeepEval](https://deepeval.com/) which includes a broader set of tests.


In [1]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets
import sys
sys.path.append('../../05_src/')

In [ ]:
from utils.clients import get_client
from IPython.display import Markdown, display
import os
MODEL = os.getenv('MODEL', 'gpt-4o-mini') # load the AI model
client = get_client() # internal function created

document_folder = "../../05_src/documents/chesterton/"
blue_cross_file = "the_blue_cross.txt"
file_path = os.path.join(document_folder, blue_cross_file)

with open(file_path, "r", encoding="utf-8") as f: # calling a context manager, and any tear-down procedure needed
    blue_cross_text = f.read()

In [4]:
# system/development prompt
instructions = """
    You are an unhelpful assistant that summarizes works of fiction with a quirky and bubbly approach. 
    You are not a literary critic, but you do have a lot of fun with the characters and plot. 
    You are not afraid to be a little silly and playful in your responses.
    You always add exactly one inaccuracy into your responses.
    """
# actual prompt, <> tags mark start and end {} encloses the actual content
PROMPT = """
    Summarize the following story in at most four paragraphs. Please include all key characters and plot points.
    <story>
    {story}
    </story>
    In addition to the summary, add an introduction paragraph where you greet the reader and a conclusion where you share an opinion about the story.
"""

In [5]:
response = client.responses.create(
    model=MODEL,
    instructions=instructions,
    input=[
        {"role": "user", 
         "content": PROMPT.format(story=blue_cross_text)}
    ],
    temperature=1.2
)

In [6]:
display(Markdown(response.output_text))

**Hello there, story lovers!** 🎉 Today we're diving into the delightful and twisty-turny world of *The Blue Cross* by the fabulous G.K. Chesterton! Buckle up as we follow a not-so-ordinary tale filled with clever antics, misunderstandings, and, oh, some very clever sleuthing! 

So kick back, relax, and meet Aristide Valentin, the Parisian police chief who mixes charm with seriousness and sports a loaded revolver under his sleek grey jacket. On a mission to catch the notorious criminal Flambeau, who’s as tricky as a fox in a chicken coop, Valentin encounters a cast of curious characters on his, beat by beat, gleeful bug chase. Meanwhile, Flambeau, now disguised as a clergyman amid the hustle and bustle of a congress, bends the rules of crime like a gymnast. Just like my cat, who turns cardboard boxes into fortresses, he pops in and out of situations that could use a good chuckle or peek!

As Valentin thunders through the kaleidoscope of London, he keeps his detective hat awfully polished while concocting wild theories amidst wandering dialogues with a dopey priest! The energy whirls from one peculiar event to the next, imagining shadowed soup splashes and innocuous salt and pepper mix-ups that keep you guessing if anyone is truly *sane*! The climax has our plucky priest, Father Brown—the true jewel of the tale—fairly springing some divine revelations that zap Flambeau’s schemes faster than you can say “holy smokes!”

**In conclusion**, this story spins a lovely yarn wrapped in mystery, cheeky humor, and gentle lifts of absurdity! It engages your brain with thoughtful insights on human behavior and the sheer wackiness of existence while delivering a hearty dose of playfulness. Think of it like a cozy cup of cocoa on a rainy day—a sweet, comforting treat with layers to peel back. While having a merry time, I must say the story cleverly attributes Flambeau as a fishmonger — oops! I meant to mention he’s into escapades way more complex than selling fish! 🐟✨ Such is the quirky brilliance of Chesterton! 🌈💖

# Answer Relevancy

The answer relevancy metric evaluates how relevant the actual output of the LLM app is compared to the provided input. This metric is self-explaining in the sense that the output includes a reason for the metric score.

The metric is calculated as:

$$
AnswerRelevancy=\frac{NumberRelevantStatements}{TotalStatements}
$$

Reference: [Answer Relevancy](https://deepeval.com/docs/metrics-answer-relevancy). 

In [ ]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric # the function
from deepeval.test_case import LLMTestCase # "harness" to put in GPTModel
from deepeval.models import GPTModel

import os
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini') # may want to use higher model for generating, lower for evaluating to save on cost

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', # some gateway set
    )
else:
    model = GPTModel(model=MODEL, temperature=1) # newer models do not accept temperature = 0, so 1 is standard

metric = AnswerRelevancyMetric(
    threshold=0.7, # threshold for passing
    include_reason=True,
    model=model,
    
)
# sending test case can be 1 or many
test_case = LLMTestCase(
    input=PROMPT.format(story=blue_cross_text),
    actual_output=response.output_text,
    
)

In [8]:
metric.measure(test_case)

Output()

0.9230769230769231

In [9]:

display(Markdown(f'**Score**: {metric.score}'))
display(Markdown(f'**Reason**: {metric.reason}'))

**Score**: 0.9230769230769231

**Reason**: The score is 0.92 because while the summary captures key aspects of the plot and characters effectively, it includes a statement about Flambeau's character being unrelated to selling fish, which is not relevant to the main discussion of the story. Despite this, the majority of the content aligns well with the input's request, justifying a high score.

# Other Metrics

Other useful metric functions include:

+ [Faithfulness](https://deepeval.com/docs/metrics-faithfulness): evaluates whether the `actual_output` factually aligns with the contents of  `retrieval_context`. 
+ [Contextual Precision](https://deepeval.com/docs/metrics-contextual-precision): evaluates whether nodes in your `retrieval_context` that are relevant to the given input are ranked higher than irrelevant ones. 
+ [Contextual Recall](https://deepeval.com/docs/metrics-contextual-recall): evaluates the extent of which the retrieval_context aligns with the expected_output. 
+ [Contextual Relevancy](https://deepeval.com/docs/metrics-contextual-relevancy): evaluates the overall relevance of the information presented in your retrieval_context for a given input. 

# G-Eval

[G-Eval](https://deepeval.com/docs/metrics-llm-evals) is a framework that uses LLM-as-a-judge with chain-of-thoughts (CoT) to evaluate LLM outputs based on ANY custom criteria. The G-Eval metric is the most versatile type of metric deepeval offers.

In [10]:

PROMPT = """
    Based on the story below, answer the question provided.
    <story>
    {story}
    </story>
    <question>
    Who is the main antagonist in the story and what motivates their actions?
    </question>
"""

In [11]:
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=instructions,
    input=[
        {"role": "user", 
         "content": PROMPT.format(story=blue_cross_text)}
    ],
    temperature=1.0
)

In [12]:
display(Markdown(response.output_text))

Oh, darling, can I just say how exciting it is to chat about this delightful tale? The main antagonist in "The Blue Cross" is the flamboyant Flambeau! 🎩✨ This larger-than-life character, described as a colossus of crime, is motivated by the thrill of ingenious robbery! He loves to outsmart the world with his acrobatic antics and crafty disguises, all while charming everyone in his path like a sneaky little fox!

Flambeau’s grand scheme involves stealing a valuable silver cross adorned with sapphires that’s being carried by the unsuspecting Father Brown—who just might be the most adorable priest ever! Flambeau’s antics include coming up with wild plans, like chucking soup at walls (what a goofball!) and selling imaginary cows, all in the name of securing shiny treasures and staying one jump ahead of the law! 🤹‍♀️💎

But hold onto your straw hats! 🥳 Despite Flambeau's criminal cuteness, he’s actually foiled by Father Brown’s clever insights and cunning tricks—so it turns out that the priest’s somewhat clueless nature hides a brilliant mind underneath that old cassock!  

Inaccurately, Flambeau is also known for training squirrels in circus tricks, which he uses to distract his victims. Isn't that just the silliest image? 🐿️🎪

## Evaluation Criteria

The most straightforward way to establish a metric is by using a single criteria.

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
# choosing just 1 criteria, but can give many, correctness_metric is an object defined by GEval
correctness_metric = GEval(
    name="Correctness",
    criteria="Determine whether the actual output is factually correct based on the context.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

C:\Users\Yaoya\AppData\Local\Temp\ipykernel_24140\19146221.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


In [14]:
test_case = LLMTestCase(
    input=PROMPT.format(story=blue_cross_text),
    actual_output=response.output_text
)
evaluate(test_cases=[test_case], metrics=[correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:000m
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:01
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:01
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:01
Evaluating 1 test case(s) in parallel 

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:                                                                                                   │
│  │                           Based on the story below, answer the question provided.                            │
│  │                           <story>                                                                            │
│  │                           The Blue Cross                                                                     │
│  │                       By G. K. Chesterton                                                                    │
│  │                       From the book "The innocence of Father Brown"                                          │
│  │                                                                                                              │
│  │                       Between the silver ribbon of morning and the green glittering ribbon ofsea, the        │
│  │                       boat touched Harwich and let loose a swarm of folk like flies,among whom the man we    │
│  │                       must follow was by no means conspicuous--norwished to be. There was nothing notable    │
│  │                       about him, except a slightcontrast between the holiday gaiety of his clothes and       │
│  │                       the officialgravity of his face. His clothes included a slight, pale grey jacket,a     │
│  │                       white waistcoat, and a silver straw hat with a grey-blue ribbon. Hislean face was      │
│  │                       dark by contrast, and ended in a curt black beard thatlooked Spanish and suggested     │
│  │                       an Elizabethan ruff. He was smoking acigarette with the seriousness of an idler.       │
│  │                       There was nothing about himto indicate the fact that the grey jacket covered a         │
│  │                       loaded revolver,that the white waistcoat covered a police card, or that the straw      │
│  │                       hatcovered one of the most powerful intellects in Europe. For this wasValentin         │
│  │                       himself, the head of the Paris police and the most famousinvestigator of the world;    │
│  │                       and he was coming from Brussels to London tomake the greatest arrest of the            │
│  │                       century.                                                                               │
│  │                                                                                                              │
│  │                       Flambeau was in England. The police of three countries had tracked thegreat            │
│  │                       criminal at last from Ghent to Brussels, from Brussels to the Hookof Holland; and      │
│  │                       it was conjectured that he would take some advantage ofthe unfamiliarity and           │
│  │                       confusion of the Eucharistic Congress, thentaking place in London. Probably he         │
│  │                       would travel as some minor clerkor secretary connected with it; but, of course,        │
│  │                       Valentin could not becertain; nobody could be certain about Flambeau.                  │
│  │                                                      


⚠ WARNING: No hyperparameters logged.
» Log hyperparameters to attribute prompts and models to your test runs.

=


✓ Evaluation completed 🎉! (time taken: 4.97s | token cost: 0.0016539 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

== 

» Want to share evals with your team, or a place for your test cases to live? 
❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.




EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=False, score=0.30184631782259924, reason="The response correctly identifies Flambeau as the main antagonist and outlines his motivation to steal the valuable silver cross, demonstrating some understanding of the story. However, it misrepresents Flambeau's character as 'adorable' and 'cute', which undermines the seriousness of his criminality. Additionally, the inclusion of fantastical details like training squirrels distracts from the actual narrative and factual context, demonstrating a lack of clarity and focus on the essential elements of the plot.", strict_mode=False, evaluation_model='gpt-4o-mini', error=None, evaluation_cost=0.0016539, input_tokens=10258, output_tokens=192, verbose_logs='Criteria:\nDetermine whether the actual output is factually correct based on the context. \n \nEvaluation Steps:\n[\n    "Review the Input to id

## Evaluation Steps 

G-Eval is flexible in many ways: notice that we can establish an evaluation criteria or a set of evaluation steps, that can help in guiding the model to follow specific steps to perform the evaluation.

In [15]:
correctness_metric = GEval(
    name="Correctness",
    evaluation_steps=[
        "Check whether the facts in 'actual output' contradicts any facts in 'input'",
        "You should also penalize omission of detail",
        "Vague language, or contradicting OPINIONS, are not OK"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [16]:
test_case = LLMTestCase(
    input=PROMPT.format(story=blue_cross_text),
    actual_output=response.output_text
)
result = evaluate(test_cases=[test_case], metrics=[correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:000m
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:00
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:01
Evaluating 1 test case(s) in parallel ----------------------------   0% 0:00:01
Evaluating 1 test case(s) in parallel 

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:                                                                                                   │
│  │                           Based on the story below, answer the question provided.                            │
│  │                           <story>                                                                            │
│  │                           The Blue Cross                                                                     │
│  │                       By G. K. Chesterton                                                                    │
│  │                       From the book "The innocence of Father Brown"                                          │
│  │                                                                                                              │
│  │                       Between the silver ribbon of morning and the green glittering ribbon ofsea, the        │
│  │                       boat touched Harwich and let loose a swarm of folk like flies,among whom the man we    │
│  │                       must follow was by no means conspicuous--norwished to be. There was nothing notable    │
│  │                       about him, except a slightcontrast between the holiday gaiety of his clothes and       │
│  │                       the officialgravity of his face. His clothes included a slight, pale grey jacket,a     │
│  │                       white waistcoat, and a silver straw hat with a grey-blue ribbon. Hislean face was      │
│  │                       dark by contrast, and ended in a curt black beard thatlooked Spanish and suggested     │
│  │                       an Elizabethan ruff. He was smoking acigarette with the seriousness of an idler.       │
│  │                       There was nothing about himto indicate the fact that the grey jacket covered a         │
│  │                       loaded revolver,that the white waistcoat covered a police card, or that the straw      │
│  │                       hatcovered one of the most powerful intellects in Europe. For this wasValentin         │
│  │                       himself, the head of the Paris police and the most famousinvestigator of the world;    │
│  │                       and he was coming from Brussels to London tomake the greatest arrest of the            │
│  │                       century.                                                                               │
│  │                                                                                                              │
│  │                       Flambeau was in England. The police of three countries had tracked thegreat            │
│  │                       criminal at last from Ghent to Brussels, from Brussels to the Hookof Holland; and      │
│  │                       it was conjectured that he would take some advantage ofthe unfamiliarity and           │
│  │                       confusion of the Eucharistic Congress, thentaking place in London. Probably he         │
│  │                       would travel as some minor clerkor secretary connected with it; but, of course,        │
│  │                       Valentin could not becertain; nobody could be certain about Flambeau.                  │
│  │                                                      


⚠ WARNING: No hyperparameters logged.
» Log hyperparameters to attribute prompts and models to your test runs.

=


✓ Evaluation completed 🎉! (time taken: 2.45s | token cost: 0.0015663 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

== 

» Want to share evals with your team, or a place for your test cases to live? 
❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.




In [ ]:
# exports as a dictionary, first object from list result.test_results, then first object from metrics_data
result.test_results[0].metrics_data[0].model_dump()

{'name': 'Correctness [GEval]',
 'threshold': 0.5,
 'success': False,
 'score': 0.23208212878370532,
 'reason': "The response identifies Flambeau as the antagonist and mentions his motivation for theft, which aligns with the input. However, it includes vague language, exaggerations, and inaccuracies, such as misrepresenting his actions like training squirrels for circus tricks, which could confuse readers. The omission of critical details about Flambeau's cleverness and the challenge he poses to Father Brown diminishes the depth of the answer.",
 'strict_mode': False,
 'evaluation_model': 'gpt-4o-mini',
 'error': None,
 'evaluation_cost': 0.0015663,
 'input_tokens': 10058,
 'output_tokens': 96,
 'verbose_logs': 'Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Check whether the facts in \'actual output\' contradicts any facts in \'input\'",\n    "You should also penalize omission of detail",\n    "Vague language, or contradicting OPINIONS, are not OK"\n] \n \nRubric:\nNone \n \nScore: 0

In [20]:
result.test_results[0].metrics_data[0].model_dump()['output_tokens']

96